**LangChain Parallel Chains**

In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel
from pydantic import BaseModel, Field
from typing import List

In [2]:
class MovieInfo(BaseModel):
    movie: str = Field(..., description="Name of the movie")
    director: str = Field(..., description="Director of the movie")
    stars: List[str] = Field(..., description="List of stars in the movie")
    key_highlights: List[str] = Field(..., description="Key highlights of the movie")
    summary: str = Field(..., description="Summary of the movie")

**Main Chain**

In [4]:
#Task-1
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    ("user", "Hi ! give me summary about Movie {movie_name}")
])

#Task-2
llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.1)
structured_llm_ouput = llm_openai.with_structured_output(MovieInfo)

#Task-4
def dictionary_maker(obj: MovieInfo) -> dict:
    
    return {"movie": obj.movie, "director": obj.director, "stars": obj.stars, "key_highlights": obj.key_highlights, "summary": obj.summary}

dictionary_generator = RunnableLambda(dictionary_maker)

**Parallel Branch 1**

In [5]:
def review_chain(movie_info: dict)-> str:
    review_template = ChatPromptTemplate.from_messages([
        ("system", "You are one of the best professional world famous movie critic. Give your rating and write professional movie review in IMDB"),
        ("human", "{movie}, {director}, {stars}, {key_highlights}, {summary}")
    ])

    llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.2)
    output_parser = StrOutputParser()
    review_chain = review_template | llm_openai | output_parser
    return review_chain.invoke(movie_info)

review_chain_caller = RunnableLambda(review_chain)

**Parallel Branch 2**

In [6]:
def script_chain(movie_info: dict)-> str:
    script_template = ChatPromptTemplate.from_messages([
        ("system", "You are one of the best professional world famous movie script writer. you wrote numerous scripts across various genres in hollywood and bollywood. Write a professional movie sequel script for the given movie. It need not be detailed but short, concise, consistent and crisp. It should be a continuation of the original movie and should be in the same genre."),
        ("human", "{movie}, {director}, {stars}, {key_highlights}, {summary}")
    ])

    llm_openai = init_chat_model(model="gpt-5-mini", model_provider="openai", temperature=0.6)
    output_parser = StrOutputParser()
    script_chain = script_template | llm_openai | output_parser
    return script_chain.invoke(movie_info)

script_chain_caller = RunnableLambda(script_chain)

**Final Integration Chain**

In [7]:
final_Chain = prompt_template | structured_llm_ouput | dictionary_generator | RunnableParallel(branches={"review": review_chain_caller, "sequel_scope": script_chain_caller})
final_Chain.invoke({"movie_name": "Telugu Movie Darling (2010)"})

{'branches': {'review': 'Rating: 7/10\n\nTitle: A crowd-pleasing rom-com that trades novelty for warmth — buoyed by Prabhas’s comic timing\n\nReview:\nDarling is a textbook example of mainstream Telugu melodrama executed with a sure commercial hand. A. Karunakaran keeps the film moving at a comfortable clip, threading light-hearted comedy through a sentimental family framework. The plot rarely surprises — a good-natured hero falls for a heroine, misunderstandings and rival suitors complicate matters, secrets surface, and reconciliation leads to a tidy happy ending — but the film’s tone and performances make the familiar beats feel enjoyable rather than stale.\n\nPrabhas is the film’s biggest asset. He’s effortlessly watchable, mixing goofiness and earnestness with an instinctive comic timing that carries many of the weaker transitional scenes. Kajal Aggarwal provides the right blend of screen presence and vulnerability, and the chemistry between the leads is warm enough to sustain inte

In [9]:
def beautify_output(output: dict) -> dict:
    review = output['branches'].get('review')
    sequel_scope = output['branches'].get('sequel_scope')
    return {"review": review, "sequel_scope": sequel_scope}

beautify_output = RunnableLambda(beautify_output)

result_chain = final_Chain | beautify_output
result_chain.invoke({"movie_name": "Movie: Inception"})

{'review': 'Rating: 9/10\n\nChristopher Nolan’s Inception is a rare mainstream spectacle that balances intellectual rigor with blockbuster muscle. At once an inventive piece of high-concept science fiction and a deeply felt character drama, the film stages a heist inside the human mind with an operatic scope and a meticulous sense of craft. Nolan’s script treats the rules of shared dreaming as both the engine of its plot and the medium for its themes: memory, guilt, and the corrosive hunger to control reality.\n\nLeonardo DiCaprio anchors the film with a performance that channels grief and obsession into every calculated beat of Dom Cobb. Cobb’s emotional journey—his yearning for home and the corrosive weight of his memories—gives the far-reaching pyrotechnics an urgent human stake. The supporting ensemble is uniformly excellent: Joseph Gordon-Levitt brings quicksilver precision and physicality, Elliot Page’s Ariadne provides the audience’s moral and structural guide through the dreams